# Notebook 1: Your First Convolutional Network (MNIST)

**Goal.** Build, train and evaluate a small convolutional neural network (CNN) that recognises handwritten digits, and understand *why* every component is there.

**What you will learn**

1. How images are represented as tensors in PyTorch.
2. Why fully connected networks are a poor fit for images, and what convolution fixes.
3. The mathematics of a convolutional layer: kernels, channels, stride, padding.
4. Non-linearities (ReLU) and pooling.
5. Softmax, cross-entropy, and how a classifier is trained with gradient descent.
6. A clean, reusable PyTorch training and evaluation loop.

**Prerequisites.** Basic Python, NumPy-style array thinking, and the idea of a derivative. No prior PyTorch knowledge is assumed.

> **Colab tip.** Go to *Runtime → Change runtime type → GPU*. Everything here also runs on CPU in a few minutes.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(0)  # reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 1. Images are tensors

A grayscale image of height $H$ and width $W$ is a matrix of pixel intensities. A colour image has three such matrices (red, green, blue), called **channels**. PyTorch stores a *batch* of images as a 4D tensor with shape

$$
(N, C, H, W) \quad = \quad (\text{batch size},\ \text{channels},\ \text{height},\ \text{width}).
$$

This layout is called **NCHW** and is what every `nn.Conv2d` layer expects.

MNIST contains 70,000 grayscale digits of size $28 \times 28$: 60,000 for training and 10,000 for testing. So one batch of 64 MNIST images has shape $(64, 1, 28, 28)$.

### Preprocessing

`transforms.ToTensor()` converts a PIL image with integer pixels in $\{0,\dots,255\}$ into a float tensor in $[0, 1]$ with shape $(C, H, W)$.

`transforms.Normalize(mean, std)` then applies $x \mapsto (x - \mu)/\sigma$ per channel. For MNIST the dataset statistics are $\mu \approx 0.1307$ and $\sigma \approx 0.3081$. Centring the inputs around zero with unit variance makes optimisation better conditioned: gradients for the first layer's weights have similar scales in every direction, so a single learning rate works well for all of them.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

full_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Hold out 5,000 training images as a validation set.
# Rule: tune on validation, report on test, and only look at test at the very end.
train_set, val_set = random_split(full_train, [len(full_train) - 5000, 5000],
                                  generator=torch.Generator().manual_seed(0))

print(len(train_set), "train |", len(val_set), "val |", len(test_set), "test")
x, y = full_train[0]
print("one image:", x.shape, x.dtype, "label:", y)

In [ ]:
# Look at the data before modelling it. Always.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, range(16)):
    img, label = full_train[i]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("MNIST samples (normalised, so the colour scale is shifted)")
plt.tight_layout()
plt.show()

### DataLoaders

A `DataLoader` wraps a dataset and yields mini-batches. Two key arguments:

* `batch_size`: how many examples per gradient step. Larger batches give less noisy gradient estimates but fewer updates per epoch.
* `shuffle=True` for training: presenting examples in a new random order every epoch keeps successive mini-batches from being correlated, which is an assumption behind stochastic gradient descent (SGD).

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print("batch of images:", xb.shape, "| batch of labels:", yb.shape)

## 2. Why not just use a fully connected network?

The simplest approach is to flatten each $28\times 28$ image into a 784-dimensional vector and feed it to a multilayer perceptron (MLP). This works on MNIST (around 98%), but it has three structural weaknesses that become fatal on real images:

1. **Parameter explosion.** A single fully connected layer from a $224\times224\times3$ image to 1,000 hidden units needs $150{,}528 \times 1{,}000 \approx 1.5\times 10^8$ weights.
2. **No notion of locality.** Flattening throws away the fact that neighbouring pixels are related. The MLP would perform equally well if we applied the same fixed random permutation to the pixels of every image, which tells us it is not using spatial structure at all.
3. **No translation equivariance.** A "7" in the top left and a "7" in the bottom right activate completely different weights. The network must learn the concept "7" separately at every location, which costs data.

Convolutional layers fix all three with two assumptions, called **inductive biases**:

* **Local connectivity.** Each output unit looks only at a small neighbourhood (for example $3\times3$) of the input.
* **Weight sharing.** The same small set of weights (the *kernel*) is used at every spatial location.

Weight sharing gives **translation equivariance**: if you shift the input, the feature map shifts in the same way. Formally, if $T_v$ translates an image by a vector $v$, then

$$
\text{Conv}(T_v\, x) = T_v\, \text{Conv}(x).
$$

(Exactly true for stride 1 and ignoring borders.) Pooling and global averaging later convert this equivariance into approximate **invariance**: the class prediction should not change when the digit moves.

## 3. The convolution operation

### 3.1 Single channel

Take an input image $X$ and a small kernel $K$ of size $k\times k$. The output (the **feature map**) at position $(i, j)$ is

$$
Y[i, j] = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} K[u, v]\; X[i + u,\ j + v] \; + \; b.
$$

In words: place the kernel on the image, multiply element-wise, sum, add a bias, then slide to the next position. (Strictly, this is *cross-correlation*; true convolution flips the kernel. Since the kernel is learned, the distinction does not matter and every deep learning library computes cross-correlation.)

A kernel acts as a **pattern detector**: the output is large where the local image patch resembles the kernel. A kernel like
$$
\begin{pmatrix} -1 & 0 & 1\\ -2 & 0 & 2\\ -1 & 0 & 1 \end{pmatrix}
$$
(the Sobel filter) responds to vertical edges. In a CNN, nobody designs the kernels: they are learned by gradient descent.

### 3.2 Multiple channels

Real layers map $C_\text{in}$ input channels to $C_\text{out}$ output channels. Each output channel $o$ has its own kernel of shape $C_\text{in}\times k \times k$, which looks at **all** input channels at once:

$$
Y[o, i, j] = b_o + \sum_{c=1}^{C_\text{in}} \sum_{u,v} K[o, c, u, v]\; X[c,\ i+u,\ j+v].
$$

So the weight tensor of `nn.Conv2d(C_in, C_out, k)` has shape $(C_\text{out}, C_\text{in}, k, k)$ and the layer has

$$
\#\text{params} = C_\text{out}\cdot C_\text{in}\cdot k^2 + C_\text{out}.
$$

Crucially this does **not** depend on the image size. That is the payoff of weight sharing.

### 3.3 Stride and padding

* **Padding** $p$: add $p$ rows/columns of zeros around the border. Without it, every $3\times3$ conv shrinks the image by 2 pixels and border pixels are under-used.
* **Stride** $s$: move the kernel $s$ pixels at a time. Stride 2 halves the spatial resolution.

The output size along one dimension is

$$
H_\text{out} = \left\lfloor \frac{H_\text{in} + 2p - k}{s} \right\rfloor + 1.
$$

With $k=3,\ p=1,\ s=1$ we get $H_\text{out} = H_\text{in}$: a **"same" convolution**. This is the most common setting, because it lets you change resolution only where you decide to (with pooling or stride).

Let's verify all of this numerically.

In [ ]:
# A hand-made 6x6 image with a vertical edge in the middle
img = torch.zeros(1, 1, 6, 6)
img[..., 3:] = 1.0
sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]]).view(1, 1, 3, 3)  # (C_out, C_in, k, k)

out = F.conv2d(img, sobel_x)            # no padding
print("input:\n", img[0, 0])
print("output (", tuple(out.shape), "):\n", out[0, 0])
print("Strong response exactly where the edge is. 6x6 -> 4x4 because p=0, k=3.")

In [ ]:
# Check the output-size formula and the parameter count formula
def out_size(h, k, s=1, p=0):
    return (h + 2 * p - k) // s + 1

for (k, s, p) in [(3, 1, 0), (3, 1, 1), (5, 1, 2), (3, 2, 1), (2, 2, 0)]:
    conv = nn.Conv2d(1, 1, kernel_size=k, stride=s, padding=p)
    measured = conv(torch.zeros(1, 1, 28, 28)).shape[-1]
    print(f"k={k} s={s} p={p}: formula={out_size(28, k, s, p):2d}, measured={measured:2d}")

conv = nn.Conv2d(16, 32, kernel_size=3)
n = sum(p.numel() for p in conv.parameters())
print("\nConv2d(16, 32, 3) params:", n, "= 32*16*9 + 32 =", 32 * 16 * 9 + 32)

## 4. Non-linearity and pooling

### 4.1 ReLU

A stack of convolutions without anything in between is still one big linear map, so depth would buy nothing. We insert a non-linearity after each conv. The standard choice is the **Rectified Linear Unit**:

$$
\text{ReLU}(z) = \max(0, z).
$$

Why ReLU rather than sigmoid or tanh? Its derivative is exactly 1 for positive inputs, so gradients pass through many layers without shrinking (sigmoid's derivative is at most $0.25$, so a 10-layer chain can scale gradients by $0.25^{10}\approx 10^{-6}$: the *vanishing gradient* problem). It is also very cheap to compute and produces sparse activations.

### 4.2 Max pooling

`nn.MaxPool2d(2)` splits each feature map into non-overlapping $2\times2$ windows and keeps the maximum of each. Effects:

* **Downsampling**: halves height and width, so the next layers are 4x cheaper.
* **Small-shift robustness**: if a feature moves by one pixel within a window, the max is unchanged.
* **Larger receptive field**: after pooling, a $3\times3$ kernel in the next layer covers a $6\times6$ region of the original image (more on receptive fields in Notebook 2).

Pooling has **no learnable parameters**.

A common design pattern follows: as spatial size goes **down**, the number of channels goes **up**. Early layers detect many simple things at high resolution (edges); later layers detect fewer-but-richer things at low resolution (strokes, loops, digit parts).

In [ ]:
x = torch.tensor([[1., 3., 2., 0.],
                  [4., 2., 1., 1.],
                  [0., 1., 5., 6.],
                  [2., 2., 7., 1.]]).view(1, 1, 4, 4)
print("input:\n", x[0, 0])
print("MaxPool2d(2):\n", nn.MaxPool2d(2)(x)[0, 0])
print("ReLU on [-2, -0.5, 0, 1.5]:", F.relu(torch.tensor([-2., -0.5, 0., 1.5])))

## 5. The model

Our architecture:

| Layer | Output shape (for one image) | Params |
|---|---|---|
| input | $1 \times 28 \times 28$ | 0 |
| Conv2d(1, 16, 3, padding=1) + ReLU | $16 \times 28 \times 28$ | $16\cdot1\cdot9+16 = 160$ |
| MaxPool2d(2) | $16 \times 14 \times 14$ | 0 |
| Conv2d(16, 32, 3, padding=1) + ReLU | $32 \times 14 \times 14$ | $32\cdot16\cdot9+32 = 4{,}640$ |
| MaxPool2d(2) | $32 \times 7 \times 7$ | 0 |
| Flatten | $1568$ | 0 |
| Linear(1568, 10) | $10$ | $15{,}690$ |

Total: about 20k parameters. For comparison, a 2-hidden-layer MLP with 256 units uses about 270k parameters for a similar or worse result.

### `nn.Module` in two sentences

You subclass `nn.Module`, create layers in `__init__` (which registers their parameters automatically), and describe the computation in `forward`. You never call `forward` directly: call `model(x)`, which also runs any registered hooks.

`nn.Sequential` is a container that just applies its layers in order. `flatten(1)` keeps dimension 0 (the batch) and flattens everything else.

The network outputs 10 raw scores called **logits**, one per class. We do **not** apply softmax inside the model, for the reason explained in the next section.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 1x28x28 -> 16x28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 16x14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # -> 32x14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 32x7x7
        )
        self.classifier = nn.Linear(32 * 7 * 7, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)          # (N, 32, 7, 7) -> (N, 1568)
        return self.classifier(x) # logits, shape (N, 10)

model = SmallCNN().to(device)
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("logits for one batch:", model(xb.to(device)).shape)

## 6. From logits to a loss: softmax and cross-entropy

The **softmax** turns logits $z \in \mathbb{R}^{K}$ into a probability distribution:

$$
p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}.
$$

Given the true class $y$, the **cross-entropy loss** is the negative log-probability assigned to it:

$$
\mathcal{L}(z, y) = -\log p_y = -z_y + \log \sum_j e^{z_j}.
$$

Interpretation: this is maximum likelihood. Minimising the average cross-entropy over the dataset is the same as maximising the probability the model assigns to the observed labels. A perfect, confident prediction gives loss $0$; a uniform guess over 10 classes gives $\log 10 \approx 2.30$, which is a useful sanity check for the loss at initialisation.

The gradient with respect to the logits is beautifully simple:

$$
\frac{\partial \mathcal{L}}{\partial z_k} = p_k - \mathbb{1}[k = y].
$$

It pushes the correct logit up and all others down, in proportion to how wrong the model is.

**Why no softmax in the model?** `nn.CrossEntropyLoss` takes logits and computes `log_softmax` + negative log-likelihood in one fused, numerically stable step (the *log-sum-exp trick*). Applying softmax yourself and then taking a log can overflow or produce `log(0) = -inf`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# Sanity check: an untrained network should give loss close to log(10)
with torch.no_grad():
    init_loss = loss_fn(model(xb.to(device)), yb.to(device)).item()
print(f"initial loss {init_loss:.3f} vs log(10) = {np.log(10):.3f}")

# Verify the gradient formula dL/dz = p - onehot(y) on a toy example
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
target = torch.tensor([0])
F.cross_entropy(z, target).backward()
print("autograd :", z.grad)
print("formula  :", F.softmax(z, dim=1).detach() - F.one_hot(target, 3))

## 7. Optimisation

Training means finding parameters $\theta$ that minimise the average loss

$$
J(\theta) = \frac{1}{|D|} \sum_{(x,y)\in D} \mathcal{L}(f_\theta(x), y).
$$

**Mini-batch SGD** estimates the gradient on a small random batch $B$ and steps downhill:

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta \frac{1}{|B|}\sum_{(x,y)\in B}\mathcal{L}(f_\theta(x), y),
$$

with learning rate $\eta$. The gradient is computed by **backpropagation**, which PyTorch's autograd does for you when you call `loss.backward()`.

We use **Adam**, which keeps running averages of the gradient (momentum) and of its square, and divides by the square root of the latter. Effectively every parameter gets its own adaptive step size, which makes Adam forgiving about the choice of $\eta$. `lr=1e-3` is a robust default.

### The five-line training step (memorise this)

```python
optimizer.zero_grad()      # 1. clear old gradients (PyTorch accumulates them by default)
logits = model(x)          # 2. forward pass
loss = loss_fn(logits, y)  # 3. compute the loss
loss.backward()            # 4. backprop: fills p.grad for every parameter
optimizer.step()           # 5. update parameters using p.grad
```

### Train mode vs eval mode

`model.train()` and `model.eval()` switch the behaviour of layers such as Dropout and BatchNorm (Notebook 3). Our model has neither yet, but building the habit now prevents painful bugs later. During evaluation we also wrap code in `torch.no_grad()` so autograd does not record the graph, saving memory and time.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n

In [ ]:
EPOCHS = 5
model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn)
    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
        history[k].append(v)
    print(f"epoch {epoch} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time() - t0:.1f}s")

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(epochs, history["train_loss"], "o-", label="train")
a1.plot(epochs, history["val_loss"], "o-", label="val")
a1.set_title("Cross-entropy loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(epochs, history["train_acc"], "o-", label="train")
a2.plot(epochs, history["val_acc"], "o-", label="val")
a2.set_title("Accuracy"); a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.show()

**How to read these curves.** If training loss keeps dropping while validation loss rises, the model is **overfitting** (memorising training examples). If both are high and flat, it is **underfitting** (too little capacity or too low a learning rate). On MNIST this small CNN typically reaches about 99% validation accuracy with the two curves close together: the inductive biases of convolution make the problem easy.

## 8. Final test evaluation and error analysis

Now, and only now, we touch the test set.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"TEST accuracy: {test_acc:.4f}  ({(1 - test_acc) * len(test_set):.0f} mistakes out of {len(test_set)})")

# Collect all predictions for analysis
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        p = F.softmax(model(x.to(device)), dim=1).cpu()
        all_probs.append(p); all_preds.append(p.argmax(1)); all_labels.append(y)
all_probs = torch.cat(all_probs); all_preds = torch.cat(all_preds); all_labels = torch.cat(all_labels)

# Confusion matrix: row = true class, column = predicted class
cm = torch.zeros(10, 10, dtype=torch.long)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1
plt.figure(figsize=(5.5, 5))
plt.imshow(cm.log1p(), cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (log colour scale)")
plt.xticks(range(10)); plt.yticks(range(10)); plt.show()

In [ ]:
# Look at the mistakes, sorted by how confident the model was
wrong = (all_preds != all_labels).nonzero().squeeze(1)
conf = all_probs[wrong, all_preds[wrong]]
order = wrong[conf.argsort(descending=True)][:16]

fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for ax, idx in zip(axes.flat, order):
    img, _ = test_set[idx]
    ax.imshow(img.squeeze(0), cmap="gray"); ax.axis("off")
    ax.set_title(f"T{all_labels[idx].item()} P{all_preds[idx].item()}\n{all_probs[idx].max():.2f}", fontsize=9)
plt.suptitle("Most confident mistakes (T = true, P = predicted, confidence)")
plt.tight_layout(); plt.show()

Many of these are genuinely ambiguous, or even mislabelled. Looking at errors is the single most informative thing you can do after training a model.

## 9. Saving and loading

Save the **`state_dict`** (a dictionary of parameter tensors), not the whole model object. It is portable across code changes as long as the layer names match.

In [ ]:
torch.save(model.state_dict(), "small_cnn_mnist.pt")

reloaded = SmallCNN().to(device)
reloaded.load_state_dict(torch.load("small_cnn_mnist.pt", map_location=device))
print("reloaded test acc:", evaluate(reloaded, test_loader, loss_fn)[1])

## 10. Optional: the MLP baseline

To *feel* the value of convolution, train an MLP with comparable effort.

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)
print("MLP parameters:", sum(p.numel() for p in mlp.parameters()))
opt_mlp = torch.optim.Adam(mlp.parameters(), lr=1e-3)
for epoch in range(1, 3):
    train_one_epoch(mlp, train_loader, opt_mlp, loss_fn)
    print(f"MLP epoch {epoch}: val acc {evaluate(mlp, val_loader, loss_fn)[1]:.4f}")

The MLP has over 10x more parameters and usually ends up slightly *worse*. On harder images (Notebook 3) the gap becomes huge.

## Summary

* Images are $(N, C, H, W)$ tensors.
* A conv layer slides learned kernels over the input; weights are shared across positions, giving few parameters and translation equivariance.
* $H_\text{out} = \lfloor (H + 2p - k)/s \rfloor + 1$; params $= C_\text{out} C_\text{in} k^2 + C_\text{out}$.
* ReLU provides non-linearity; max pooling downsamples and adds small-shift robustness.
* The model outputs logits; `nn.CrossEntropyLoss` applies log-softmax internally.
* Training step: `zero_grad → forward → loss → backward → step`. Use `model.train()` / `model.eval()` and `torch.no_grad()` for evaluation.

## Exercises

1. **Learning rate sweep.** Retrain with `lr` in `{1e-1, 1e-2, 1e-3, 1e-4}`. Plot validation accuracy per epoch for each. What happens at `1e-1`?
2. **Optimizer.** Replace Adam with `torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)`. Compare convergence speed.
3. **Remove pooling.** Delete both `MaxPool2d` layers and fix the `Linear` input size. How do parameter count, speed and accuracy change?
4. **Wider vs deeper.** Try (a) 64 and 128 channels; (b) a third conv block. Which gives the better accuracy per parameter?
5. **Translation test.** Shift every test image by 4 pixels with `torch.roll(x, shifts=4, dims=-1)` and re-evaluate both the CNN and the MLP. Which degrades more, and why? (Hint: Section 2.)
6. **Stretch.** Implement the 2D convolution of Section 3.1 with explicit Python loops for a single channel, and check it matches `F.conv2d` to within `1e-6`.